# 관계와 측정값을 함께 확인하기

제공 과제 코드·Neo4j·PostgreSQL의 실제 결과를 확인합니다. 인용 존재와 승인·조치 완료를 구분합니다.

In [1]:
import sys,json,hashlib
from pathlib import Path
import pandas as pd
sys.path.insert(0,'/course/플랫폼코드')
import graph_lab as lab
import context_queries as practical
from hydops.b4_ontology import graph
from hydops.b3_tsdb import store

def graph_hash():
    nodes=graph.run('MATCH (n) RETURN elementId(n) AS id, labels(n) AS labels, properties(n) AS properties ORDER BY id')
    rels=graph.run('MATCH ()-[r]->() RETURN elementId(r) AS id, type(r) AS type, properties(r) AS properties ORDER BY id')
    return hashlib.sha256(json.dumps([nodes,rels],sort_keys=True,default=str).encode()).hexdigest()
before=graph_hash()
event_id='EVT-20270126-090043-HYD-01-e51b'
event=lab.run_read(lab.Q_EVENT_EVIDENCE,event_id=event_id)
assert len(event)==1 and len(event[0]['citations'])>0, '사전 인용 사건을 먼저 확인하세요.'
print('준비 사건:',event_id)
print('설비:',event[0]['asset_id'],'/ 유형:',event[0]['event_type'],'/ 상태:',event[0]['status'])


준비 사건: EVT-20270126-090043-HYD-01-e51b
설비: HYD-01 / 유형: COOLING_ANOMALY / 상태: PENDING_APPROVAL


In [2]:
# 사건이 인용한 문서의 판과 절
citations=graph.run("""MATCH (e:Event {event_id:$id})-[:CITES]->(sec:SOPSection)<-[:HAS_SECTION]-(sop:SOP)
RETURN sop.doc_id AS document,sop.version AS version,sec.section_no AS section,sec.heading AS heading
ORDER BY document,version,section""",id=event_id)
assert len(citations)==len(event[0]['citations'])
display(pd.DataFrame(citations).rename(columns={'document':'문서 ID','version':'판','section':'절','heading':'제목'}))
print('사건 상태:',event[0]['status'],'· 인용 조회만 수행했습니다.')


,문서 ID,판,절,제목
0,SOP-COOL-001,2,2,발동 조건
1,SOP-COOL-001,2,4,허용 조치
2,SOP-VER-001,1,2,발동 조건


사건 상태: PENDING_APPROVAL · 인용 조회만 수행했습니다.


In [3]:
# 같은 ID와 잘못된 ID의 차이
for sensor in graph.SENSORS:
    current=graph.run('MATCH (s:Sensor {sensor_id:$id}) RETURN properties(s) AS p',id='HYD-01.'+sensor['code'])
    assert len(current)==1 and all(current[0]['p'].get(k)==v for k,v in sensor.items()), '기존 센서 변경을 보존해야 하므로 시험 중지'
same=lab.seed_check()
assert len(same)==4 and all(r['committed'] and r['nodes_created']==r['relationships_created']==0 for r in same)
display(pd.DataFrame([{'대상':r['what'],'새 노드':r['nodes_created'],'새 관계':r['relationships_created'],'반영':r['committed']} for r in same]))
assert graph.run('MATCH (s:Sensor {sensor_id:$id}) RETURN count(s) AS n',id='HYD-01-TS1')[0]['n']==0
wrong=lab.merge_if_same_as_seed(lab.MERGE_SENSOR.replace("+ '.' +", "+ '-' +"),asset_id='HYD-01',**graph.SENSORS[0])
remaining=graph.run('MATCH (s:Sensor {sensor_id:$id}) RETURN count(s) AS n',id='HYD-01-TS1')[0]['n']
assert wrong['nodes_created']==1 and wrong['relationships_created']==1 and not wrong['committed'] and remaining==0
display(pd.DataFrame([{'시험 ID':'HYD-01-TS1','생성될 노드':wrong['nodes_created'],'생성될 관계':wrong['relationships_created'],
                      '반영':wrong['committed'],'되돌린 뒤 노드':remaining}]))


,대상,새 노드,새 관계,반영
0,Sensor HYD-01/TS1,0,0,True
1,Sensor HYD-01/PS1,0,0,True
2,Sensor HYD-01/FS1,0,0,True
3,SOP-COOL-001@v2 APPLIES_TO HYD-01,0,0,True


,시험 ID,생성될 노드,생성될 관계,반영,되돌린 뒤 노드
0,HYD-01-TS1,1,1,False,0


In [4]:
# 그래프의 센서와 DB의 최근 관측 연결
run_id=lab.load_lab_cycle()
try:
    joined=lab.sensors_with_recent_values('HYD-01',run_id)
    practical_result=practical.join_recent('HYD-01',run_id)
    assert len(joined)==3 and all(r['n']==r['valid']==60 and r['origin_cycle_id']==100 for r in joined)
    assert all(r['unit_graph']==r['unit_db'] for r in joined)
    assert len(practical_result['sensors'])==3 and all(r['samples']==60 and r['unit_match'] for r in practical_result['sensors'])
    print('실습 실행:',run_id,'/ HYD-01 / UCI 사이클100')
    display(pd.DataFrame(joined).rename(columns={'sensor_id':'그래프 센서 ID','unit_graph':'그래프 단위','unit_db':'DB 단위',
          'n':'관측 수','valid':'품질 OK','last_value':'마지막 원시값','origin_cycle_id':'원본 사이클'}))
    wrong_key=store.recent_window('HYD-01',60,sensor_id='HYD-01.TS1',run_id=run_id)
    right_key=store.recent_window('HYD-01',60,sensor_id='TS1',run_id=run_id)
    assert len(wrong_key)==0 and len(right_key)==60
    display(pd.DataFrame([{'설비 조건':'HYD-01','센서 조건':key,'조회 행':n} for key,n in [('HYD-01.TS1',len(wrong_key)),('TS1',len(right_key))]]))
    _ = Path('관계와최근값.json').write_text(json.dumps({'run_id':run_id,'integrated':joined,'practical':practical_result},ensure_ascii=False,indent=2,default=str),encoding='utf-8')
finally:
    lab.cleanup(run_id)


실습 실행: LAB-S03-392af1be / HYD-01 / UCI 사이클100


,그래프 센서 ID,그래프 단위,DB 단위,관측 수,품질 OK,마지막 원시값,원본 사이클
0,HYD-01.FS1,L/min,L/min,60,60,7.884,100
1,HYD-01.PS1,bar,bar,60,60,147.345,100
2,HYD-01.TS1,°C,°C,60,60,53.395,100


,설비 조건,센서 조건,조회 행
0,HYD-01,HYD-01.TS1,0
1,HYD-01,TS1,60


In [5]:
# 결과 파일 다시 열기와 내 임시 실행 정리
saved=json.loads(Path('관계와최근값.json').read_text(encoding='utf-8'))
assert saved['integrated']==joined and saved['run_id']==run_id
with store.connect() as connection:
    left=connection.execute('SELECT count(*) AS n FROM observation WHERE run_id=%s',(run_id,)).fetchone()['n']
    run_left=connection.execute('SELECT count(*) AS n FROM run WHERE run_id=%s',(run_id,)).fetchone()['n']
assert left==run_left==0
after=graph_hash()
assert before==after, '그래프 전후 내용이 달라졌습니다. 원인 확인 필요.'
assert event==lab.run_read(lab.Q_EVENT_EVIDENCE,event_id=event_id)
print('다시 읽은 결과:',len(saved['integrated']),'센서 / 각각60개 관측 / 원본 사이클100')
print('정리한 실습 실행:',run_id)
print('DB에 남은 해당 실행:',run_left,'/ 관측:',left)
print('기존 사건 상태·인용 동일 / 그래프 전후 내용 해시 동일')
record={'event':event,'citations':citations,'same_ids':same,'wrong_id':wrong,'wrong_id_after':remaining,
        'joined':joined,'run_id':run_id,'run_remaining':run_left,'observations_remaining':left,
        'saved_result_reopened':True,'graph_before_sha256':before,'graph_after_sha256':after}
_ = Path('검증결과.json').write_text(json.dumps(record,ensure_ascii=False,indent=2,default=str),encoding='utf-8')


다시 읽은 결과: 3 센서 / 각각60개 관측 / 원본 사이클100
정리한 실습 실행: LAB-S03-392af1be
DB에 남은 해당 실행: 0 / 관측: 0
기존 사건 상태·인용 동일 / 그래프 전후 내용 해시 동일
